In [6]:
from pathlib import Path
import sys

# Set project directory
ROOT = Path(r"C:\Users\User\Desktop\ML\MachineLearning_CW2")

# Allow importing project modules
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

# Import shared data preparation
from src.data_preparation import load_training_bundle

# Load training data
X_train, y_train, metadata = load_training_bundle(ROOT)

# Display dataset information
print("Training Features:", X_train.shape)
print("Training Target:", y_train.shape)
print("Metadata:", metadata.shape)

print("\nTarget Distribution:")
print(y_train.value_counts())

Training Features: (202795, 21)
Training Target: (202795,)
Metadata: (202795, 4)

Target Distribution:
HeartDiseaseorAttack
0    183698
1     19097
Name: count, dtype: int64


In [7]:
from sklearn.linear_model import LogisticRegression
from src.model_setup import make_model_pipeline, shared_cv

# Create Logistic Regression baseline model
lr_model = make_model_pipeline(
    LogisticRegression(
        max_iter=2000,
        random_state=42
    ),
    kind="linear"
)

# Use shared 5-fold cross-validation
cv = shared_cv(metadata)

print("Logistic Regression Baseline Model Created")
print("\nModel Pipeline:")
print(lr_model)

print("\nNumber of CV Splits:", cv.get_n_splits())

Logistic Regression Baseline Model Created

Model Pipeline:
Pipeline(steps=[('preprocess',
                 ColumnTransformer(transformers=[('numeric',
                                                  Pipeline(steps=[('imputer',
                                                                   SimpleImputer(strategy='median')),
                                                                  ('scaler',
                                                                   StandardScaler())]),
                                                  ['BMI', 'MentHlth',
                                                   'PhysHlth']),
                                                 ('binary',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  ['HighBP', 'HighChol',
                                                   'CholCheck', 'Smoker',
                                                   'Stroke', 'PhysActi

In [8]:
import pandas as pd
from sklearn.model_selection import cross_validate

# Evaluation metrics
scoring = {
    "Accuracy": "accuracy",
    "Precision": "precision",
    "Recall": "recall",
    "F1": "f1",
    "ROC_AUC": "roc_auc",
    "Average_Precision": "average_precision"
}

# Train and evaluate using shared 5-fold CV
lr_cv_results = cross_validate(
    estimator=lr_model,
    X=X_train,
    y=y_train,
    cv=cv,
    scoring=scoring,
    n_jobs=1,
    return_train_score=True
)

# Summarise validation results
lr_summary = pd.DataFrame({
    "Metric": list(scoring.keys()),
    "Mean Validation Score": [
        lr_cv_results[f"test_{metric}"].mean()
        for metric in scoring
    ],
    "Standard Deviation": [
        lr_cv_results[f"test_{metric}"].std()
        for metric in scoring
    ]
})

print("Logistic Regression Baseline - 5-Fold Cross-Validation")
display(lr_summary.round(4))

Logistic Regression Baseline - 5-Fold Cross-Validation


,Metric,Mean Validation Score,Standard Deviation
0,Accuracy,0.9074,0.0012
1,Precision,0.5360,0.0192
2,Recall,0.1236,0.0050
3,F1,0.2009,0.0078
4,ROC_AUC,0.8465,0.0039
5,Average_Precision,0.3687,0.0065


In [9]:
from pathlib import Path

# Create results folder
results_dir = ROOT / "results" / "member2"
results_dir.mkdir(parents=True, exist_ok=True)

# Save Logistic Regression baseline results
lr_summary.to_csv(
    results_dir / "logistic_regression_baseline.csv",
    index=False
)

print("Logistic Regression Baseline Results Saved Successfully")

Logistic Regression Baseline Results Saved Successfully


In [10]:
# Compare training and validation performance
overfitting_results = pd.DataFrame({
    "Metric": ["Accuracy", "F1", "ROC_AUC", "Average_Precision"],
    "Training Score": [
        lr_cv_results[f"train_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ],
    "Validation Score": [
        lr_cv_results[f"test_{metric}"].mean()
        for metric in ["Accuracy", "F1", "ROC_AUC", "Average_Precision"]
    ]
})

overfitting_results["Gap"] = (
    overfitting_results["Training Score"]
    - overfitting_results["Validation Score"]
)

display(overfitting_results.round(4))

,Metric,Training Score,Validation Score,Gap
0,Accuracy,0.9075,0.9074,0.0001
1,F1,0.2011,0.2009,0.0002
2,ROC_AUC,0.8470,0.8465,0.0005
3,Average_Precision,0.3693,0.3687,0.0007


In [11]:
# Save overfitting results
overfitting_results.to_csv(
    results_dir / "logistic_regression_overfitting.csv",
    index=False
)

print("Logistic Regression Overfitting Results Saved Successfully")

Logistic Regression Overfitting Results Saved Successfully
